In [114]:
# ============================================================
# CAREERCOMPASS — CAREER LOCATION RECOMMENDATION ENGINE
# ============================================================
#
# Goal:
# Build a simple, interpretable system that compares locations
# for a selected occupation using BLS labor-market data.
#
# ------------------------------------------------------------
# STATE-LEVEL ANALYSIS
# ------------------------------------------------------------
# State recommendations are based on three state-specific
# labor-market measures:
#
# 1. Median annual wage
# 2. Total employment
# 3. Location quotient (LQ)
#
# Each measure is ranked within the same occupation across
# states and converted to a percentile score.
#
# The three scores are then combined using equal weighting:
#
# CareerCompass Score =
# (Wage Score + Employment Score + LQ Score) / 3
#
# The resulting score is used to rank states for each occupation.
#
# ------------------------------------------------------------
# MSA-LEVEL ANALYSIS
# ------------------------------------------------------------
# The same methodology is applied at the metropolitan-area level.
#
# For each occupation, MSAs are compared using:
#
# 1. Median annual wage
# 2. Total employment
# 3. Location quotient (LQ)
#
# The three percentile scores are equally weighted to produce
# an MSA CareerCompass Score, which is then used to rank MSAs.
#
# ------------------------------------------------------------
# CAREER OUTLOOK INFORMATION
# ------------------------------------------------------------
# National Employment Projections are joined to both the State
# and MSA datasets to provide additional occupation-level context:
#
# - Projected employment growth, 2025–2035
# - Average annual occupational openings
# - Typical education required
# - Related work experience
# - Typical on-the-job training
# - Occupational Outlook Handbook (OOH) content
#
# These national projection measures are NOT used to distinguish
# one state or MSA from another because the projections are
# occupation-level rather than location-specific.
#
# They are retained as career-outlook information for the
# recommendation and RAG layers.
#
# ------------------------------------------------------------
# MISSING DATA
# ------------------------------------------------------------
# Original BLS data is preserved.
# Missing wage, employment, or LQ values are not replaced with 0.
# If one or more scoring components are unavailable, the
# CareerCompass Score remains NaN and SCORE_AVAILABLE is False.
#
# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------
# State:
# - CareerCompass Score
# - State Rank
# - Score Availability
#
# MSA:
# - CareerCompass Score
# - MSA Rank
# - Score Availability
#
# The resulting datasets will be used by the CareerCompass
# Streamlit application and RAG layer.
# ============================================================

In [1]:
import pandas as pd
import numpy as np

state = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\state_detailed.xlsx")
national = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\national_detailed.xlsx")
msa = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\msa_detailed.xlsx")
ep = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\employment_projections_detailed.xlsx")
openings = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\openings_detailed.xlsx")
bos = pd.read_excel(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\bos_detailed.xlsx")

print(state.shape)
print(national.shape)
print(msa.shape)
print(ep.shape)
print(openings.shape)
print(bos.shape)

(36168, 34)
(830, 32)
(141015, 34)
(831, 17)
(831, 14)
(45542, 34)


In [2]:
print("State columns: ", state.columns)
print("National columns: ", national.columns)
print("Msa columns: ", msa.columns)
print("Ep columns: ", ep.columns)
print("Opening columns: ", openings.columns)
print("Bos columns: ", bos.columns)

State columns:  Index(['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE',
       'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP',
       'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
       'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN',
       'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75',
       'A_PCT90', 'ANNUAL', 'HOURLY', 'H_WAGE_ABOVE_THRESHOLD',
       'A_WAGE_ABOVE_THRESHOLD'],
      dtype='object')
National columns:  Index(['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE',
       'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP',
       'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
       'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN',
       'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75',
       'A_PCT90', 'ANNUAL', 'HOURLY'],
      dtype='object')
Msa columns:  Index(['AREA', '

In [3]:
state_cc = state[["AREA", "AREA_TITLE", "PRIM_STATE", "OCC_CODE", "OCC_TITLE", "TOT_EMP", "LOC_QUOTIENT",
"A_MEDIAN"]].copy()

state_cc = state_cc.rename(columns={"AREA": "STATE_AREA_CODE", "AREA_TITLE": "STATE", "PRIM_STATE": "STATE_CODE",
"TOT_EMP": "STATE_EMPLOYMENT", "LOC_QUOTIENT": "LOCATION_QUOTIENT", "A_MEDIAN": "STATE_MEDIAN_WAGE"})

In [4]:
national_cc = national[["OCC_CODE", "OCC_TITLE", "TOT_EMP", "A_MEDIAN"]].copy()

national_cc = national_cc.rename(columns={"TOT_EMP": "NATIONAL_EMPLOYMENT",
"A_MEDIAN": "NATIONAL_MEDIAN_WAGE"})

In [5]:
ep_cc = ep[["2025 National Employment Matrix code", "Employment, 2025", "Employment, 2035",
"Employment change, numeric, 2025–35", "Employment change, percent, 2025–35",
"Occupational openings, 2025–35 annual average", "Median annual wage, dollars, 2025",
"Typical education needed for entry", "Work experience in a related occupation",
"Typical on-the-job training needed to attain competency in the occupation",
"Related Occupational Outlook Handbook (OOH) content"]].copy()

ep_cc = ep_cc.rename(columns={"2025 National Employment Matrix code": "OCC_CODE",
"Employment, 2025": "EP_EMPLOYMENT_2025", "Employment, 2035": "EP_EMPLOYMENT_2035",
"Employment change, numeric, 2025–35": "EMPLOYMENT_CHANGE",
"Employment change, percent, 2025–35": "PROJECTED_GROWTH_PCT",
"Occupational openings, 2025–35 annual average": "ANNUAL_OPENINGS",
"Median annual wage, dollars, 2025": "EP_MEDIAN_WAGE",
"Typical education needed for entry": "EDUCATION",
"Work experience in a related occupation": "RELATED_EXPERIENCE",
"Typical on-the-job training needed to attain competency in the occupation": "TRAINING",
"Related Occupational Outlook Handbook (OOH) content": "OOH_CONTENT"})

In [6]:
careercompass_state = state_cc.merge(national_cc, on="OCC_CODE", how="left")

In [7]:
careercompass_state = careercompass_state.merge(ep_cc, on="OCC_CODE",how="left")

In [8]:
careercompass_state.shape

(36168, 21)

In [9]:
careercompass_state.head()

,STATE_AREA_CODE,STATE,STATE_CODE,OCC_CODE,OCC_TITLE_x,STATE_EMPLOYMENT,LOCATION_QUOTIENT,STATE_MEDIAN_WAGE,OCC_TITLE_y,NATIONAL_EMPLOYMENT,...,EP_EMPLOYMENT_2025,EP_EMPLOYMENT_2035,EMPLOYMENT_CHANGE,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,EP_MEDIAN_WAGE,EDUCATION,RELATED_EXPERIENCE,TRAINING,OOH_CONTENT
0,1,Alabama,AL,11-1011,Chief Executives,810.0,0.29,166420.0,Chief Executives,204350,...,291.6,300.9,9.3,3.2,19.1,213990.0,Bachelor's degree,5 years or more,NaN,OOH Content
1,1,Alabama,AL,11-1021,General and Operations Managers,30170.0,0.63,111790.0,General and Operations Managers,3503020,...,3599.0,3780.3,181.3,5.0,285.0,105770.0,Bachelor's degree,5 years or more,NaN,OOH Content
2,1,Alabama,AL,11-2011,Advertising and Promotions Managers,60.0,0.22,64640.0,Advertising and Promotions Managers,21470,...,33.2,31.9,-1.2,-3.7,2.3,133660.0,Bachelor's degree,Less than 5 years,NaN,OOH Content
3,1,Alabama,AL,11-2021,Marketing Managers,1540.0,0.29,120160.0,Marketing Managers,395240,...,421.6,450.8,29.1,6.9,34.0,166790.0,Bachelor's degree,5 years or more,NaN,OOH Content
4,1,Alabama,AL,11-2022,Sales Managers,4670.0,0.54,124040.0,Sales Managers,637080,...,650.1,679.1,29.0,4.5,47.3,148270.0,Bachelor's degree,Less than 5 years,NaN,OOH Content


In [10]:
careercompass_state.isnull().sum()

STATE_AREA_CODE             0
STATE                       0
STATE_CODE                  0
OCC_CODE                    0
OCC_TITLE_x                 0
STATE_EMPLOYMENT         1318
LOCATION_QUOTIENT        1318
STATE_MEDIAN_WAGE         624
OCC_TITLE_y                 0
NATIONAL_EMPLOYMENT         0
NATIONAL_MEDIAN_WAGE      147
EP_EMPLOYMENT_2025          0
EP_EMPLOYMENT_2035          0
EMPLOYMENT_CHANGE           0
PROJECTED_GROWTH_PCT        0
ANNUAL_OPENINGS             0
EP_MEDIAN_WAGE            147
EDUCATION                   0
RELATED_EXPERIENCE      30914
TRAINING                14797
OOH_CONTENT                 0
dtype: int64

In [11]:
careercompass_state = careercompass_state.drop(columns=["OCC_TITLE_y"])
careercompass_state = careercompass_state.rename(columns={"OCC_TITLE_x": "OCC_TITLE"})

In [12]:
careercompass_state[["STATE_MEDIAN_WAGE", "STATE_EMPLOYMENT", "PROJECTED_GROWTH_PCT", "ANNUAL_OPENINGS"]].isnull().sum()

STATE_MEDIAN_WAGE        624
STATE_EMPLOYMENT        1318
PROJECTED_GROWTH_PCT       0
ANNUAL_OPENINGS            0
dtype: int64

In [13]:
careercompass_scoring = careercompass_state[
["STATE", "STATE_CODE", "OCC_CODE", "OCC_TITLE", "STATE_MEDIAN_WAGE", "STATE_EMPLOYMENT",
"LOCATION_QUOTIENT", "PROJECTED_GROWTH_PCT", "ANNUAL_OPENINGS"]].copy()

In [14]:
careercompass_scoring.head()

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS
0,Alabama,AL,11-1011,Chief Executives,166420.0,810.0,0.29,3.2,19.1
1,Alabama,AL,11-1021,General and Operations Managers,111790.0,30170.0,0.63,5.0,285.0
2,Alabama,AL,11-2011,Advertising and Promotions Managers,64640.0,60.0,0.22,-3.7,2.3
3,Alabama,AL,11-2021,Marketing Managers,120160.0,1540.0,0.29,6.9,34.0
4,Alabama,AL,11-2022,Sales Managers,124040.0,4670.0,0.54,4.5,47.3


In [15]:
careercompass_scoring["WAGE_SCORE"] = (careercompass_scoring.groupby("OCC_CODE")["STATE_MEDIAN_WAGE"].rank(pct=True) * 100)

In [16]:
careercompass_scoring["EMPLOYMENT_SCORE"] = (careercompass_scoring.groupby("OCC_CODE")["STATE_EMPLOYMENT"].rank(pct=True) * 100)

In [17]:
careercompass_scoring["LQ_SCORE"] = (careercompass_scoring.groupby("OCC_CODE")["LOCATION_QUOTIENT"].rank(pct=True) * 100)

In [18]:
careercompass_scoring.head()

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE
0,Alabama,AL,11-1011,Chief Executives,166420.0,810.0,0.29,3.2,19.1,27.083333,28.260870,8.695652
1,Alabama,AL,11-1021,General and Operations Managers,111790.0,30170.0,0.63,5.0,285.0,70.370370,38.888889,17.592593
2,Alabama,AL,11-2011,Advertising and Promotions Managers,64640.0,60.0,0.22,-3.7,2.3,2.631579,11.428571,8.571429
3,Alabama,AL,11-2021,Marketing Managers,120160.0,1540.0,0.29,6.9,34.0,18.000000,34.000000,15.000000
4,Alabama,AL,11-2022,Sales Managers,124040.0,4670.0,0.54,4.5,47.3,28.301887,45.098039,20.588235


In [22]:
careercompass_scoring["CAREERCOMPASS_SCORE"] = (careercompass_scoring["WAGE_SCORE"] 
+ careercompass_scoring["EMPLOYMENT_SCORE"] + careercompass_scoring["LQ_SCORE"]) / 3

careercompass_scoring["CAREERCOMPASS_SCORE"] = (careercompass_scoring["CAREERCOMPASS_SCORE"].round(2))

In [23]:
careercompass_scoring[careercompass_scoring["OCC_TITLE"] == "Data Scientists"].sort_values("CAREERCOMPASS_SCORE",
ascending=False).head(10)

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE,CAREERCOMPASS_SCORE
32662,Washington,WA,15-2051,Data Scientists,163350.0,9600.0,1.60,34.6,24.8,100.0,86.274510,98.039216,94.77
2721,California,CA,15-2051,Data Scientists,141590.0,39310.0,1.28,34.6,24.8,98.0,100.000000,80.392157,92.80
22114,New York,NY,15-2051,Data Scientists,130460.0,23970.0,1.47,34.6,24.8,90.0,96.078431,92.156863,92.75
14420,Massachusetts,MA,15-2051,Data Scientists,131750.0,9420.0,1.53,34.6,24.8,92.0,84.313725,96.078431,90.80
22898,North Carolina,NC,15-2051,Data Scientists,119090.0,11430.0,1.37,34.6,24.8,74.0,92.156863,89.215686,85.12
29855,Texas,TX,15-2051,Data Scientists,122090.0,25860.0,1.09,34.6,24.8,76.0,98.039216,72.549020,82.20
31905,Virginia,VA,15-2051,Data Scientists,126430.0,8920.0,1.29,34.6,24.8,82.0,80.392157,83.333333,81.91
26465,Pennsylvania,PA,15-2051,Data Scientists,106850.0,13810.0,1.35,34.6,24.8,62.0,94.117647,86.274510,80.80
5493,District of Columbia,DC,15-2051,Data Scientists,126490.0,2680.0,2.27,34.6,24.8,84.0,52.941176,100.000000,78.98
3525,Colorado,CO,15-2051,Data Scientists,117400.0,6280.0,1.29,34.6,24.8,72.0,74.509804,83.333333,76.61


In [24]:
careercompass_scoring["CAREERCOMPASS_SCORE"].isnull().sum() #checking for scores where any of the three metrics is Nan

np.int64(1932)

In [25]:
careercompass_scoring["CAREERCOMPASS_SCORE"].notnull().sum()  #checking for scores where all three metrics present

np.int64(34236)

In [26]:
#Adding a flag column for rows where any of the three metrics are missing
careercompass_scoring["SCORE_AVAILABLE"] = (careercompass_scoring["CAREERCOMPASS_SCORE"].notnull())

In [27]:
careercompass_scoring["SCORE_AVAILABLE"].value_counts()

SCORE_AVAILABLE
True     34236
False     1932
Name: count, dtype: int64

In [28]:
careercompass_scoring.head()

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE,CAREERCOMPASS_SCORE,SCORE_AVAILABLE
0,Alabama,AL,11-1011,Chief Executives,166420.0,810.0,0.29,3.2,19.1,27.083333,28.260870,8.695652,21.35,True
1,Alabama,AL,11-1021,General and Operations Managers,111790.0,30170.0,0.63,5.0,285.0,70.370370,38.888889,17.592593,42.28,True
2,Alabama,AL,11-2011,Advertising and Promotions Managers,64640.0,60.0,0.22,-3.7,2.3,2.631579,11.428571,8.571429,7.54,True
3,Alabama,AL,11-2021,Marketing Managers,120160.0,1540.0,0.29,6.9,34.0,18.000000,34.000000,15.000000,22.33,True
4,Alabama,AL,11-2022,Sales Managers,124040.0,4670.0,0.54,4.5,47.3,28.301887,45.098039,20.588235,31.33,True


In [29]:
#Creating a function for state wise career recommendations
def get_state_recommendations(occupation):
    result = careercompass_scoring[careercompass_scoring["OCC_TITLE"] == occupation].copy()
    
    result = result[result["SCORE_AVAILABLE"] == True]
    
    result = result.sort_values("CAREERCOMPASS_SCORE",ascending=False)
    
    return result

In [30]:
#Testing the function
get_state_recommendations("Data Scientists").head(10)

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE,CAREERCOMPASS_SCORE,SCORE_AVAILABLE
32662,Washington,WA,15-2051,Data Scientists,163350.0,9600.0,1.60,34.6,24.8,100.0,86.274510,98.039216,94.77,True
2721,California,CA,15-2051,Data Scientists,141590.0,39310.0,1.28,34.6,24.8,98.0,100.000000,80.392157,92.80,True
22114,New York,NY,15-2051,Data Scientists,130460.0,23970.0,1.47,34.6,24.8,90.0,96.078431,92.156863,92.75,True
14420,Massachusetts,MA,15-2051,Data Scientists,131750.0,9420.0,1.53,34.6,24.8,92.0,84.313725,96.078431,90.80,True
22898,North Carolina,NC,15-2051,Data Scientists,119090.0,11430.0,1.37,34.6,24.8,74.0,92.156863,89.215686,85.12,True
29855,Texas,TX,15-2051,Data Scientists,122090.0,25860.0,1.09,34.6,24.8,76.0,98.039216,72.549020,82.20,True
31905,Virginia,VA,15-2051,Data Scientists,126430.0,8920.0,1.29,34.6,24.8,82.0,80.392157,83.333333,81.91,True
26465,Pennsylvania,PA,15-2051,Data Scientists,106850.0,13810.0,1.35,34.6,24.8,62.0,94.117647,86.274510,80.80,True
5493,District of Columbia,DC,15-2051,Data Scientists,126490.0,2680.0,2.27,34.6,24.8,84.0,52.941176,100.000000,78.98,True
3525,Colorado,CO,15-2051,Data Scientists,117400.0,6280.0,1.29,34.6,24.8,72.0,74.509804,83.333333,76.61,True


In [31]:
#Dense ranking states based on careercompass score
careercompass_scoring["STATE_RANK"] = (careercompass_scoring.groupby("OCC_CODE")["CAREERCOMPASS_SCORE"].rank(
ascending=False, method="dense"))

In [32]:
state_recommendations = careercompass_scoring[["STATE","STATE_CODE",
"OCC_CODE", "OCC_TITLE", "STATE_MEDIAN_WAGE", "STATE_EMPLOYMENT", "LOCATION_QUOTIENT",
"PROJECTED_GROWTH_PCT", "ANNUAL_OPENINGS", "CAREERCOMPASS_SCORE", "STATE_RANK", "SCORE_AVAILABLE"]].copy()

In [34]:
state_recommendations.head()

,STATE,STATE_CODE,OCC_CODE,OCC_TITLE,STATE_MEDIAN_WAGE,STATE_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,CAREERCOMPASS_SCORE,STATE_RANK,SCORE_AVAILABLE
0,Alabama,AL,11-1011,Chief Executives,166420.0,810.0,0.29,3.2,19.1,21.35,41.0,True
1,Alabama,AL,11-1021,General and Operations Managers,111790.0,30170.0,0.63,5.0,285.0,42.28,32.0,True
2,Alabama,AL,11-2011,Advertising and Promotions Managers,64640.0,60.0,0.22,-3.7,2.3,7.54,32.0,True
3,Alabama,AL,11-2021,Marketing Managers,120160.0,1540.0,0.29,6.9,34.0,22.33,37.0,True
4,Alabama,AL,11-2022,Sales Managers,124040.0,4670.0,0.54,4.5,47.3,31.33,37.0,True


In [83]:
#Building the careercompass msa dataset now
msa_cc = msa[["AREA", "AREA_TITLE", "PRIM_STATE", "OCC_CODE", "OCC_TITLE", "TOT_EMP",
"LOC_QUOTIENT", "A_MEDIAN"]].copy()

In [85]:
msa_cc.columns

Index(['AREA', 'AREA_TITLE', 'PRIM_STATE', 'OCC_CODE', 'OCC_TITLE', 'TOT_EMP',
       'LOC_QUOTIENT', 'A_MEDIAN'],
      dtype='object')

In [86]:
msa_cc = msa_cc.rename(columns={"AREA": "MSA_AREA_CODE",
"AREA_TITLE": "MSA", "PRIM_STATE": "STATE_CODE", "TOT_EMP": "MSA_EMPLOYMENT",
"LOC_QUOTIENT": "LOCATION_QUOTIENT", "A_MEDIAN": "MSA_MEDIAN_WAGE"})

msa_cc.columns

Index(['MSA_AREA_CODE', 'MSA', 'STATE_CODE', 'OCC_CODE', 'OCC_TITLE',
       'MSA_EMPLOYMENT', 'LOCATION_QUOTIENT', 'MSA_MEDIAN_WAGE'],
      dtype='object')

In [87]:
msa_cc.head()

,MSA_AREA_CODE,MSA,STATE_CODE,OCC_CODE,OCC_TITLE,MSA_EMPLOYMENT,LOCATION_QUOTIENT,MSA_MEDIAN_WAGE
0,10180,"Abilene, TX",TX,11-1011,Chief Executives,40.0,0.41,165770.0
1,10180,"Abilene, TX",TX,11-1021,General and Operations Managers,2120.0,1.25,86200.0
2,10180,"Abilene, TX",TX,11-2021,Marketing Managers,140.0,0.73,109100.0
3,10180,"Abilene, TX",TX,11-2022,Sales Managers,280.0,0.90,111020.0
4,10180,"Abilene, TX",TX,11-2032,Public Relations Managers,40.0,1.10,93710.0


In [88]:
ep_cc.columns

Index(['OCC_CODE', 'EP_EMPLOYMENT_2025', 'EP_EMPLOYMENT_2035',
       'EMPLOYMENT_CHANGE', 'PROJECTED_GROWTH_PCT', 'ANNUAL_OPENINGS',
       'EP_MEDIAN_WAGE', 'EDUCATION', 'RELATED_EXPERIENCE', 'TRAINING',
       'OOH_CONTENT'],
      dtype='object')

In [89]:
msa_cc = msa_cc.merge(ep_cc, on="OCC_CODE", how="left")
msa_cc.columns

Index(['MSA_AREA_CODE', 'MSA', 'STATE_CODE', 'OCC_CODE', 'OCC_TITLE',
       'MSA_EMPLOYMENT', 'LOCATION_QUOTIENT', 'MSA_MEDIAN_WAGE',
       'EP_EMPLOYMENT_2025', 'EP_EMPLOYMENT_2035', 'EMPLOYMENT_CHANGE',
       'PROJECTED_GROWTH_PCT', 'ANNUAL_OPENINGS', 'EP_MEDIAN_WAGE',
       'EDUCATION', 'RELATED_EXPERIENCE', 'TRAINING', 'OOH_CONTENT'],
      dtype='object')

In [90]:
msa_cc.head()

,MSA_AREA_CODE,MSA,STATE_CODE,OCC_CODE,OCC_TITLE,MSA_EMPLOYMENT,LOCATION_QUOTIENT,MSA_MEDIAN_WAGE,EP_EMPLOYMENT_2025,EP_EMPLOYMENT_2035,EMPLOYMENT_CHANGE,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,EP_MEDIAN_WAGE,EDUCATION,RELATED_EXPERIENCE,TRAINING,OOH_CONTENT
0,10180,"Abilene, TX",TX,11-1011,Chief Executives,40.0,0.41,165770.0,291.6,300.9,9.3,3.2,19.1,213990.0,Bachelor's degree,5 years or more,NaN,OOH Content
1,10180,"Abilene, TX",TX,11-1021,General and Operations Managers,2120.0,1.25,86200.0,3599.0,3780.3,181.3,5.0,285.0,105770.0,Bachelor's degree,5 years or more,NaN,OOH Content
2,10180,"Abilene, TX",TX,11-2021,Marketing Managers,140.0,0.73,109100.0,421.6,450.8,29.1,6.9,34.0,166790.0,Bachelor's degree,5 years or more,NaN,OOH Content
3,10180,"Abilene, TX",TX,11-2022,Sales Managers,280.0,0.90,111020.0,650.1,679.1,29.0,4.5,47.3,148270.0,Bachelor's degree,Less than 5 years,NaN,OOH Content
4,10180,"Abilene, TX",TX,11-2032,Public Relations Managers,40.0,1.10,93710.0,80.8,85.4,4.6,5.7,6.1,146910.0,Bachelor's degree,5 years or more,NaN,OOH Content


In [91]:
#Checking for nulls
msa_cc.isnull().sum()

MSA_AREA_CODE                0
MSA                          0
STATE_CODE                   0
OCC_CODE                     0
OCC_TITLE                    0
MSA_EMPLOYMENT            2924
LOCATION_QUOTIENT         2924
MSA_MEDIAN_WAGE           1699
EP_EMPLOYMENT_2025           0
EP_EMPLOYMENT_2035           0
EMPLOYMENT_CHANGE            0
PROJECTED_GROWTH_PCT         0
ANNUAL_OPENINGS              0
EP_MEDIAN_WAGE             262
EDUCATION                    0
RELATED_EXPERIENCE      117390
TRAINING                 60482
OOH_CONTENT                  0
dtype: int64

In [92]:
#Checking for duplicates
msa_cc.duplicated().sum()

np.int64(0)

In [93]:
#Creating scoring table
msa_scoring = msa_cc[["MSA", "STATE_CODE", "OCC_CODE", "OCC_TITLE", "MSA_MEDIAN_WAGE",
"MSA_EMPLOYMENT", "LOCATION_QUOTIENT", "PROJECTED_GROWTH_PCT", "ANNUAL_OPENINGS"]].copy()

In [94]:
#Creating MSA Wage score
msa_scoring["WAGE_SCORE"] = (msa_scoring.groupby("OCC_CODE")["MSA_MEDIAN_WAGE"].rank(pct=True) * 100)

In [95]:
#Creating MSA Employment score
msa_scoring["EMPLOYMENT_SCORE"] = (msa_scoring.groupby("OCC_CODE")["MSA_EMPLOYMENT"].rank(pct=True) * 100)

In [97]:
#Creating MSA location Quotient score
msa_scoring["LQ_SCORE"] = (msa_scoring.groupby("OCC_CODE")["LOCATION_QUOTIENT"].rank(pct=True) * 100)

In [98]:
#Creating MSA Careercompass score
msa_scoring["CAREERCOMPASS_SCORE"] = (msa_scoring["WAGE_SCORE"]
+ msa_scoring["EMPLOYMENT_SCORE"] + msa_scoring["LQ_SCORE"]) / 3

In [99]:
msa_scoring["CAREERCOMPASS_SCORE"] = (msa_scoring["CAREERCOMPASS_SCORE"].round(2))

In [100]:
msa_scoring["SCORE_AVAILABLE"] = (msa_scoring["CAREERCOMPASS_SCORE"].notnull())

In [101]:
msa_scoring["SCORE_AVAILABLE"].value_counts()

SCORE_AVAILABLE
True     136416
False      4599
Name: count, dtype: int64

In [102]:
msa_scoring["MSA_RANK"] = (msa_scoring.groupby("OCC_CODE")["CAREERCOMPASS_SCORE"]
.rank(ascending=False, method="dense"))

In [104]:
msa_scoring.head()

,MSA,STATE_CODE,OCC_CODE,OCC_TITLE,MSA_MEDIAN_WAGE,MSA_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE,CAREERCOMPASS_SCORE,SCORE_AVAILABLE,MSA_RANK
0,"Abilene, TX",TX,11-1011,Chief Executives,165770.0,40.0,0.41,3.2,19.1,25.394322,8.024691,11.419753,14.95,True,293.0
1,"Abilene, TX",TX,11-1021,General and Operations Managers,86200.0,2120.0,1.25,5.0,285.0,24.362245,51.020408,89.795918,55.06,True,134.0
2,"Abilene, TX",TX,11-2021,Marketing Managers,109100.0,140.0,0.73,6.9,34.0,8.734940,48.787879,77.727273,45.08,True,185.0
3,"Abilene, TX",TX,11-2022,Sales Managers,111020.0,280.0,0.90,4.5,47.3,21.336761,51.558442,80.909091,51.27,True,177.0
4,"Abilene, TX",TX,11-2032,Public Relations Managers,93710.0,40.0,1.10,5.7,6.1,6.629834,14.088398,75.690608,32.14,True,131.0


In [105]:
#Creating a function for career recommendations based on MSAs
def get_msa_recommendations(occupation):
    
    result = msa_scoring[msa_scoring["OCC_TITLE"] == occupation].copy()
    result = result[result["SCORE_AVAILABLE"] == True]
    result = result.sort_values("CAREERCOMPASS_SCORE", ascending=False)
    
    return result

get_msa_recommendations("Data Scientists").head(10)

,MSA,STATE_CODE,OCC_CODE,OCC_TITLE,MSA_MEDIAN_WAGE,MSA_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,WAGE_SCORE,EMPLOYMENT_SCORE,LQ_SCORE,CAREERCOMPASS_SCORE,SCORE_AVAILABLE,MSA_RANK
113897,"San Francisco-Oakland-Fremont, CA",CA,15-2051,Data Scientists,170110.0,10460.0,2.61,34.6,24.8,99.645390,99.652778,99.305556,99.53,True,1.0
114593,"San Jose-Sunnyvale-Santa Clara, CA",CA,15-2051,Data Scientists,185080.0,6060.0,3.16,34.6,24.8,100.000000,96.527778,99.652778,98.73,True,2.0
118396,"Seattle-Tacoma-Bellevue, WA",WA,15-2051,Data Scientists,164740.0,8370.0,2.38,34.6,24.8,98.936170,98.263889,98.958333,98.72,True,3.0
134227,"Washington-Arlington-Alexandria, DC-VA-MD-WV",DC,15-2051,Data Scientists,132200.0,9260.0,1.75,34.6,24.8,94.680851,98.611111,96.527778,96.61,True,4.0
88254,"New York-Newark-Jersey City, NY-NJ",NY,15-2051,Data Scientists,135980.0,23160.0,1.45,34.6,24.8,96.808511,100.000000,92.534722,96.45,True,5.0
23379,"Charlotte-Concord-Gastonia, NC-SC",NC,15-2051,Data Scientists,132460.0,4420.0,1.93,34.6,24.8,95.035461,95.833333,97.916667,96.26,True,6.0
15874,"Boston-Cambridge-Newton, MA-NH",MA,15-2051,Data Scientists,132040.0,7930.0,1.74,34.6,24.8,94.326241,97.569444,96.180556,96.03,True,7.0
31970,"Dallas-Fort Worth-Arlington, TX",TX,15-2051,Data Scientists,127750.0,10120.0,1.48,34.6,24.8,90.780142,99.305556,93.055556,94.38,True,8.0
8548,"Austin-Round Rock-San Marcos, TX",TX,15-2051,Data Scientists,127360.0,3730.0,1.71,34.6,24.8,89.716312,94.791667,95.833333,93.45,True,9.0
113008,"San Diego-Chula Vista-Carlsbad, CA",CA,15-2051,Data Scientists,130990.0,2830.0,1.09,34.6,24.8,93.617021,93.055556,87.152778,91.28,True,10.0


In [106]:
msa_recommendations = msa_scoring[["MSA", "STATE_CODE", "OCC_CODE", "OCC_TITLE",
"MSA_MEDIAN_WAGE", "MSA_EMPLOYMENT", "LOCATION_QUOTIENT", "PROJECTED_GROWTH_PCT",
"ANNUAL_OPENINGS", "CAREERCOMPASS_SCORE", "MSA_RANK", "SCORE_AVAILABLE"]].copy()

In [107]:
msa_recommendations.head()

,MSA,STATE_CODE,OCC_CODE,OCC_TITLE,MSA_MEDIAN_WAGE,MSA_EMPLOYMENT,LOCATION_QUOTIENT,PROJECTED_GROWTH_PCT,ANNUAL_OPENINGS,CAREERCOMPASS_SCORE,MSA_RANK,SCORE_AVAILABLE
0,"Abilene, TX",TX,11-1011,Chief Executives,165770.0,40.0,0.41,3.2,19.1,14.95,293.0,True
1,"Abilene, TX",TX,11-1021,General and Operations Managers,86200.0,2120.0,1.25,5.0,285.0,55.06,134.0,True
2,"Abilene, TX",TX,11-2021,Marketing Managers,109100.0,140.0,0.73,6.9,34.0,45.08,185.0,True
3,"Abilene, TX",TX,11-2022,Sales Managers,111020.0,280.0,0.90,4.5,47.3,51.27,177.0,True
4,"Abilene, TX",TX,11-2032,Public Relations Managers,93710.0,40.0,1.10,5.7,6.1,32.14,131.0,True


In [35]:
#Saving the state final file
state_recommendations.to_csv(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\careercompass_state.csv", index=False)

In [110]:
#Saving the MSA final file
msa_recommendations.to_csv(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\careercompass_msa.csv", index=False)

In [36]:
#Saving the state master table
careercompass_state.to_csv(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\careercompass_state_master.csv", index=False)

In [112]:
#Saving the MSA master table
msa_cc.to_csv(r"C:\Users\admin\Desktop\BLS Project\cleaned\detailed\careercompass_msa_master.csv", index=False)